# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import os
import json
import numpy as np
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.system("git clone https://github.com/PrathamDudani/FlyRank_Assignment.git")
    os.chdir("FlyRank_Assignment")

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)
os.makedirs("docs/assets", exist_ok=True)

pd.set_option("display.max_columns", 60)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)


(30000, 44)


## 1. Question

*The research question and the decision it supports.*

**Research question:** among a client's existing content pages, which ones are worth a human
reviewer's time to check for a refresh first?

**The decision this supports:** a content editor or SEO reviewer has limited hours. This work
turns raw traffic/engagement signals into a ranked, reason-coded queue so that time goes to the
pages most worth checking, instead of a random or purely manual scan. It does not decide *what*
to change on a page, and it does not publish anything — see Limitations and the Ranked
Recommendations sections for exactly where the line is drawn.


In [17]:
print("Rows in starter dataset:", len(df))
print("Clients:", df["client_id"].nunique())
print("Columns:", len(df.columns))


Rows in starter dataset: 30000
Clients: 32
Columns: 44


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

- **Release used:** the anonymized **starter slice** shipped in this repo —
  `data/raw/content_refresh_anonymized.csv` — 30,000 rows, one per pseudonymized content item,
  covering trailing-90-day metrics.
- **Scope note, stated plainly:** the FlyRank internship also offers a much larger hosted
  release (`hf://datasets/FlyRank/internship-warehouse`, roughly 79M rows across 104 clients,
  Jan 2025-Jun 2026). Week 3 of this project explored that release's schema directly (see
  `work/notebooks/w03_data_contract.ipynb`) to understand the data contract, but every model,
  audit, and playbook in this paper runs on the **30,000-row starter slice only**. That's a
  disclosed scope decision, not an oversight — see Limitations.
- **Excluded rows:** rows where `avg_position == 0` (1,205 of 30,000) are dropped throughout,
  because `avg_position = 0` means "no ranking data," not rank zero — including it would corrupt
  every position-based feature.
- **Public-safe by construction:** the starter slice ships with no titles, URLs, client names, or
  keywords — only pseudonymous `client_id` / `content_id` values and numeric metrics. Nothing in
  this paper or its source notebooks needed to touch anything more identifying than that.


In [18]:
valid = df[df["avg_position"] > 0].copy()
print(f"Rows after excluding avg_position == 0: {len(valid):,} (dropped {len(df) - len(valid):,})")
print(f"Clients remaining: {valid['client_id'].nunique()} of {df['client_id'].nunique()}")


Rows after excluding avg_position == 0: 28,795 (dropped 1,205)
Clients remaining: 31 of 32


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label.** `is_declining_label` = 1 when `trend_direction == "down"`, where `trend_direction`
is itself computed from `trend_pct`, the percent change between `impressions_last_30d` and
`impressions_prev_30d`. Those three columns are therefore never used as features — confirmed
mechanically (not just asserted) in `work/notebooks/w06_validation_audit.ipynb`.

**Features (6, all pre-label window or static):** `ctr`, `avg_position`, `impressions_90d`,
`engagement_rate`, `days_since_last_update`, `search_volume`.

**Baseline.** A transparent rule from `w04_baseline_score.ipynb`: flag a page when its CTR sits
in the worst quartile of the gap between its own CTR and the *average* CTR for its position
bucket (1-3, 4-6, 7-10, 11-20, 20+), learned from training clients only.

**Model.** Logistic regression, `class_weight="balanced"`, on the 6 features above.

**Validation design — the core methodological decision of this paper.** Rows aren't
independent: many rows share a `client_id`. A naive random row split lets the model see other
pages from the *same* client during training, which inflates every metric without teaching the
model anything that generalizes. This paper uses a **client-grouped split**
(`GroupShuffleSplit`, `random_state=42`, 20% test) — no client appears in both train and test.
The starter dataset has no per-row timestamp, so a time-aware split isn't available here; grouped
is the correct honest choice given what the data actually contains.

**Leakage checks run** (full detail in `w06_validation_audit.ipynb`): (1) mechanically confirmed
the label's formula rather than trusting its description; (2) deliberately added the
label-derived columns back into the feature set and watched AUC jump toward 1.0, proving the test
harness itself would catch a real leak; (3) checked whether `impressions_90d` — which
structurally overlaps the 60-day window the label is built from — was inflating the score (it
was not, meaningfully); (4) confirmed no pre-existing "product flag" or decision-score columns
exist in this dataset to leak in.


In [11]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

valid["is_declining_label"] = (valid["trend_direction"] == "down").astype(int)
features = ["ctr", "avg_position", "impressions_90d", "engagement_rate",
            "days_since_last_update", "search_volume"]

# Naive random split (BEFORE) -- reproduced from w06 for the paper's before/after figure
train_naive, test_naive = train_test_split(
    valid, test_size=0.2, random_state=42, stratify=valid["is_declining_label"]
)
m_naive = LogisticRegression(max_iter=1000, class_weight="balanced")
m_naive.fit(train_naive[features].fillna(0), train_naive["is_declining_label"])
auc_naive = roc_auc_score(
    test_naive["is_declining_label"],
    m_naive.predict_proba(test_naive[features].fillna(0))[:, 1],
)
client_overlap = len(set(train_naive["client_id"]) & set(test_naive["client_id"]))

# Client-grouped split (AFTER) -- the split used for every result in this paper
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(valid, groups=valid["client_id"]))
train_grouped, test_grouped = valid.iloc[train_idx].copy(), valid.iloc[test_idx].copy()

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(train_grouped[features].fillna(0), train_grouped["is_declining_label"])
proba = model.predict_proba(test_grouped[features].fillna(0))[:, 1]
pred = model.predict(test_grouped[features].fillna(0))
auc_grouped = roc_auc_score(test_grouped["is_declining_label"], proba)

print(f"Naive random-split AUC:   {auc_naive:.3f}  (clients leaking across train/test: "
      f"{client_overlap} of {valid['client_id'].nunique()})")
print(f"Client-grouped-split AUC: {auc_grouped:.3f}  (the number this paper reports)")


Naive random-split AUC:   0.566  (clients leaking across train/test: 30 of 31)
Client-grouped-split AUC: 0.542  (the number this paper reports)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Both the model and the baseline rule are scored on the **identical client-grouped holdout**
(7 clients, 5,821 rows, never seen in training) — an apples-to-apples comparison, not two
separately-tuned numbers.


In [12]:
bins = [0, 3, 6, 10, 20, 1000]
labels = ["1-3", "4-6", "7-10", "11-20", "20+"]
train_grouped["bucket"] = pd.cut(train_grouped["avg_position"], bins=bins, labels=labels).astype(str)
test_grouped["bucket"] = pd.cut(test_grouped["avg_position"], bins=bins, labels=labels).astype(str)
expected_ctr = train_grouped.groupby("bucket")["ctr"].mean().to_dict()
train_grouped["expected_ctr"] = train_grouped["bucket"].map(expected_ctr).astype(float)
test_grouped["expected_ctr"] = test_grouped["bucket"].map(expected_ctr).astype(float)
train_gap = train_grouped["expected_ctr"] - train_grouped["ctr"]
gap_threshold = train_gap.quantile(0.75)
test_grouped["ctr_gap"] = test_grouped["expected_ctr"] - test_grouped["ctr"]
baseline_pred = (test_grouped["ctr_gap"] > gap_threshold).astype(int)
baseline_auc = roc_auc_score(test_grouped["is_declining_label"], test_grouped["ctr_gap"].fillna(0))

results_table = pd.DataFrame([
    {
        "system": "Baseline rule (CTR gap)",
        "auc": round(baseline_auc, 3),
        "precision": round(precision_score(test_grouped["is_declining_label"], baseline_pred), 3),
        "recall": round(recall_score(test_grouped["is_declining_label"], baseline_pred), 3),
        "f1": round(f1_score(test_grouped["is_declining_label"], baseline_pred), 3),
    },
    {
        "system": "Model (logistic regression)",
        "auc": round(auc_grouped, 3),
        "precision": round(precision_score(test_grouped["is_declining_label"], pred), 3),
        "recall": round(recall_score(test_grouped["is_declining_label"], pred), 3),
        "f1": round(f1_score(test_grouped["is_declining_label"], pred), 3),
    },
])
base_rate = test_grouped["is_declining_label"].mean()
print(f"Base rate on this holdout: {base_rate:.3f}  (always read next to the metrics below)")
print(results_table.to_string(index=False))

results_table.to_json("work/outputs/capstone_results.json", orient="records", indent=2)


Base rate on this holdout: 0.540  (always read next to the metrics below)
                     system   auc  precision  recall    f1
    Baseline rule (CTR gap) 0.521      0.569   0.399 0.469
Model (logistic regression) 0.542      0.576   0.667 0.618


**Reading the table honestly.** The base rate is 0.540 — meaning a coin flip already gets
54% of labels right, so an AUC of 0.542 is a real but modest signal, not a strong one. The model
beats the baseline rule on every metric here, most clearly on recall (0.667 vs 0.399) and F1
(0.618 vs 0.469); AUC and precision are close between the two. The honest summary: the model adds
value mainly by *catching more of the actual declining pages* than the simple rule does, not by
being dramatically more accurate overall.


## 5. Limitations

*What this work cannot claim.*

- **Scope:** trained and validated on the 30,000-row anonymized starter slice, not the full
  79M-row warehouse release. Nothing here has been checked against that larger, more diverse
  dataset.
- **Weak solo model signal:** AUC 0.542 against a 0.540 base rate is close to chance. The model
  is useful combined with the transparent rule and human review — not as a standalone predictor.
- **Cross-sectional, not causal:** every number here comes from a single anonymized snapshot with
  no intervention. "This page is flagged" is an observed pattern, never a claim that refreshing
  it *will* recover traffic.
- **Client-held-out, not universally validated:** the grouped split proves the model generalizes
  to clients unlike the 24 training clients *within this dataset's 31 clients* — it says nothing
  about a client from an unrepresented industry, language, or content type.
- **Snapshot, not live:** scores go stale as real traffic moves. See the monitoring triggers in
  the Ranked Recommendations section.
- **Known failure pattern** (from `w06_validation_audit.ipynb`'s error analysis): the model's
  most confident false positives are stale, near-zero-traffic pages; its most confident false
  negatives are large, actively-updated pages that are actually declining. A reviewer should not
  treat a low score as reassurance on a high-traffic page.


In [13]:
# The honest failure pattern, reproduced on this exact grouped split.
errors = test_grouped.copy()
errors["pred"] = pred
errors["proba"] = proba
fp = errors[(errors["pred"] == 1) & (errors["is_declining_label"] == 0)]
fn = errors[(errors["pred"] == 0) & (errors["is_declining_label"] == 1)]
print(f"False positives: {len(fp)} of {len(test_grouped)} | False negatives: {len(fn)} of {len(test_grouped)}")
print("Median impressions_90d, false positives (model wrongly confident):", fp["impressions_90d"].median())
print("Median impressions_90d, false negatives (model wrongly reassured):", fn["impressions_90d"].median())


False positives: 1546 of 5821 | False negatives: 1046 of 5821
Median impressions_90d, false positives (model wrongly confident): 58.5
Median impressions_90d, false negatives (model wrongly reassured): 1184.0


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Full detail in `work/notebooks/w07_action_playbook.ipynb`. Five rule-based archetypes (explicitly
not statistical clusters), each mapped to one action — ranked by archetype priority, then by
traffic at stake within each tier:

| Archetype | Action | Meaning |
|---|---|---|
| Visible & Declining | `PRIORITY_REFRESH_REVIEW` | Model + rule agree, and real traffic is at stake |
| Visible & Stale | `SCHEDULE_REFRESH` | Quiet update history on a page that still gets traffic |
| Striking-Distance CTR Gap | `REVIEW_SNIPPET_CTR` | Ranks well, but CTR trails its position peers |
| Quiet / Low Visibility | `MONITOR_ONLY` | Little traffic at stake either way |
| Stable | `NO_ACTION` | No flags raised |

**Cost/value, in one line:** cheapest-and-most-visible first — clear `REVIEW_SNIPPET_CTR` before
committing hours to full `PRIORITY_REFRESH_REVIEW` rewrites, and treat priority picks as
candidates to sample-check, not execute wholesale, given the model's modest AUC.

**What must NOT be automated:** no auto-publishing or auto-editing; no automated
depublishing/pruning; no client-facing claims of causation; no fully automated scheduling that
skips human sign-off; no use of this queue for individual staff evaluation.

**Monitoring/retrain triggers:** base-rate drift beyond ~10 points from 0.540; holdout AUC
decaying below 0.542; any new client scored without its own held-out check first; feature-median
drift from the recorded training thresholds; re-run at least quarterly.


In [14]:
with open("work/outputs/w07_metrics.json") as f:
    playbook_snapshot = json.load(f)
print(json.dumps(playbook_snapshot, indent=2))


{
  "base_rate": 0.5399,
  "holdout_auc": 0.542,
  "visibility_threshold_impressions_90d": 928.0,
  "staleness_threshold_days_since_update": 22.0,
  "gap_threshold_ctr_points": 0.4305036524413686,
  "n_train_clients": 24,
  "n_test_clients": 7
}


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Two new charts for the paper (model-vs-baseline, and the split before/after), plus the two
action-queue charts already produced in Week 7, all saved to `work/figures/` (committed) and
copied to `docs/assets/` for the deployed page.


In [15]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shutil

# Chart 1: model vs baseline, same split
fig, ax = plt.subplots(figsize=(7, 4.2))
metrics = ["auc", "precision", "recall", "f1"]
x = np.arange(len(metrics))
width = 0.35
baseline_vals = results_table.loc[results_table["system"].str.contains("Baseline"), metrics].values[0]
model_vals = results_table.loc[results_table["system"].str.contains("Model"), metrics].values[0]
ax.bar(x - width/2, baseline_vals, width, label="Baseline rule", color="#8C6BB1")
ax.bar(x + width/2, model_vals, width, label="Model", color="#426B69")
ax.axhline(base_rate, color="#999", linestyle="--", linewidth=1, label=f"Base rate ({base_rate:.2f})")
ax.set_xticks(x)
ax.set_xticklabels([m.upper() if m == "auc" else m.title() for m in metrics])
ax.set_ylim(0, 1)
ax.set_title("Model vs. baseline rule, identical client-grouped holdout")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("work/figures/capstone_model_vs_baseline.png", dpi=150)
plt.close(fig)

# Chart 2: naive vs grouped split, before/after
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Naive random split\n(BEFORE)", "Client-grouped split\n(AFTER)"],
       [auc_naive, auc_grouped], color=["#B07AA1", "#4E79A7"])
ax.axhline(base_rate, color="#999", linestyle="--", linewidth=1)
ax.set_ylim(0, 1)
ax.set_ylabel("AUC")
ax.set_title("Why the split matters: same model, same data, different honesty")
fig.tight_layout()
fig.savefig("work/figures/capstone_split_before_after.png", dpi=150)
plt.close(fig)

# Copy all four charts into docs/assets for the deployed page
for name in ["capstone_model_vs_baseline.png", "capstone_split_before_after.png",
             "w07_action_mix.png", "w07_reason_codes.png"]:
    shutil.copy(f"work/figures/{name}", f"docs/assets/{name}")

print("Wrote and copied 4 chart files.")
print(os.listdir("docs/assets"))


Wrote and copied 4 chart files.
['w07_action_mix.png', 'capstone_model_vs_baseline.png', 'w07_reason_codes.png', 'capstone_split_before_after.png']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.